# 0. CONTEXTO DE LA EMPRESA Y CARGA DEL DATASET: AIRBnB

Airbnb es una plataforma en línea y aplicación móvil que conecta a viajeros con anfitriones locales que ofrecen alojamiento temporal, como habitaciones, departamentos o casas enteras, a menudo a precios más accesibles que los hoteles. Funciona bajo un modelo de economía colaborativa, permitiendo reservar experiencias personalizadas en cualquier lugar del mundo y garantizando la seguridad mediante reseñas mutuas entre huéspedes y anfitriones.

## ¿QUÉ ES AIRBNB?

Es un marketplace digital que conecta a dos tipos de usuarios:

- **Hosts (Anfitriones):** Personas que ofrecen sus propiedades (desde una habitación compartida hasta una mansión o un castillo) para alquiler a corto o largo plazo.
- **Guests (Huéspedes):** Viajeros que buscan alojamiento con una experiencia más local o económica que un hotel tradicional.

## El Modelo de Negocio

Airbnb no es dueño de las propiedades; su valor reside en la plataforma y los datos. Sus ingresos provienen de:

- **Comisiones por servicio:** Cobran un porcentaje tanto al anfitrión como al huésped por cada reserva confirmada.
- **Servicios adicionales:** "Experiencias" (tours o actividades guiadas) y servicios de gestión para anfitriones.

---

## Objetivo del Análisis

Este notebook analiza el dataset de Airbnb de Nueva York con tres ejes principales:

1. **Modelo de Regresión Lineal Múltiple** — Predecir el precio a partir de características físicas de la propiedad (`accommodates`, `bedrooms`, `beds`, `bathrooms`).
2. **Análisis Geográfico** — Identificar zonas populares, precios promedio por distrito y el comportamiento del mercado por vecindario.
3. **Análisis de Reputación del Anfitrión** — Explorar si variables como verificación de identidad, foto de perfil, reserva instantánea y reseñas se relacionan con el precio.

In [ ]:
# ============================================================================
# ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ============================================================================
# Dataset: Airbnb Price Prediction - New York City
# Fuente: https://www.kaggle.com/datasets/stevezhenghp/airbnb-price-prediction
# Estructura: ~74,000 listados de Airbnb en NYC con 29 variables que describen
#             características del anfitrión, ubicación, tipo de habitación,
#             precio y métricas de popularidad (reseñas, disponibilidad).
# ============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# INSTALAR Y DESCARGAR DATASET Y LIBRERIAS
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# 1. Definir la ruta local al archivo
# '../' sube un nivel a la carpeta principal, luego entra a 'data'
ruta_local = os.path.join('..', 'data', 'Train.csv') 

print(f"\nCargando dataset local desde: {ruta_local}")

# 2. Leer el archivo CSV
if os.path.exists(ruta_local):
    datos = pd.read_csv(ruta_local)
    print("¡Dataset cargado con éxito desde tu carpeta local!")
    display(datos.head(3))
else:
    print(f"❌ Error: No se encontró el archivo en {ruta_local}")
    print("Asegúrate de que el archivo se llame 'Train.csv' y esté dentro de la carpeta 'data'.")

In [ ]:
# ===========================================================================
# TIPOS DE DATOS Y VALORES NULOS (ANTES DE LIMPIEZA)
# ===========================================================================
print("=" * 65)
print("TIPOS DE DATOS Y VALORES NULOS — ESTADO INICIAL")
print("=" * 65)
info_df = pd.DataFrame({
    'Tipo': datos.dtypes,
    'No_Nulos': datos.notna().sum(),
    'Nulos': datos.isna().sum(),
    '% Nulos': (datos.isna().sum() / len(datos) * 100).round(2)
})
print(info_df.to_string())

## 0.1 CANTIDAD DE DATOS CONSEGUIDOS

El dataset contiene **74,111 registros** y **29 variables** originales, dando un volumen total de aproximadamente 2,146,000 datos. Se aprecian valores nulos en variables clave como `first_review`, `host_response_rate` y `review_scores_rating`, los cuales serán tratados en la fase de limpieza.

---

## 0.2 🔡 DESCRIPCIÓN DE LAS VARIABLES CATEGÓRICAS

#### Nominales
- **id:** Identificador único del registro.
- **property_type:** Tipo de propiedad (Apartamento, Casa, Loft, etc.).
- **room_type:** Tipo de habitación (Habitación privada, Casa completa, Habitación compartida).
- **amenities:** Lista de servicios disponibles (Wifi, Cocina, Aire acondicionado).
- **city:** Ciudad donde se encuentra el alojamiento.
- **neighbourhood:** Barrio o colonia específica.
- **thumbnail_url:** Enlace a la imagen miniatura del anuncio.
- **zipcode:** Código postal.

#### Binarias
- **cleaning_fee:** Si se cobra tarifa de limpieza.
- **host_has_profile_pic:** Si el anfitrión tiene foto de perfil.
- **host_identity_verified:** Si el anfitrión ha verificado su identidad.
- **instant_bookable:** Si permite la reserva inmediata.

#### Temporales (Fechas)
- **first_review / last_review:** Fechas de la primera y última reseña.
- **host_since:** Fecha en la que el anfitrión se unió a la plataforma.

---

## 0.3 🔢 DESCRIPCIÓN DE LAS VARIABLES NUMÉRICAS

- **log_price:** Logaritmo del precio (normaliza la distribución del costo).
- **accommodates:** Capacidad máxima de personas.
- **bathrooms:** Número de baños.
- **latitude / longitude:** Coordenadas geográficas.
- **number_of_reviews:** Cantidad total de reseñas recibidas.
- **review_scores_rating:** Calificación promedio (0 a 100).
- **bedrooms:** Cantidad de habitaciones.
- **beds:** Cantidad de camas físicas.

# 1. LIMPIEZA DE DATOS

In [ ]:
# ===========================================================================
# LIMPIEZA DE DATOS
# ===========================================================================
print("\nLIMPIEZA DE DATOS")

# Guardar copia del original para referencia
datos_original = datos.copy()

# 1. Eliminar duplicados
n_antes = len(datos)
datos = datos.drop_duplicates()
print(f"  Duplicados eliminados: {n_antes - len(datos)}")

# 2. Estandarizar nombres de columnas
datos.columns = datos.columns.str.strip().str.lower()

# 3. Conversión y limpieza de variables específicas
# host_response_rate: eliminar el símbolo '%' y convertir a float
if 'host_response_rate' in datos.columns:
    datos['host_response_rate'] = (
        datos['host_response_rate']
        .astype(str).str.replace('%', '', regex=False).str.strip()
    )
    datos['host_response_rate'] = pd.to_numeric(datos['host_response_rate'], errors='coerce')

# Variables booleanas: 't'/'f' → 1/0
bool_cols = ['host_has_profile_pic', 'host_identity_verified',
             'instant_bookable', 'cleaning_fee']
for col in bool_cols:
    if col in datos.columns:
        datos[col] = datos[col].map({'t': 1, 'f': 0, True: 1, False: 0})
        datos[col] = pd.to_numeric(datos[col], errors='coerce')

# 4. Imputación de variables numéricas con mediana (más robusta que media ante outliers)
numericas = datos.select_dtypes(include=np.number).columns
for col in numericas:
    if col in datos.columns:
        datos[col] = pd.to_numeric(datos[col], errors='coerce')
        if datos[col].isnull().sum() > 0:
            mediana = datos[col].median()
            datos[col] = datos[col].fillna(mediana)

# 5. Imputación de variables categóricas con moda
categoricas = datos.select_dtypes(include='object').columns
for col in categoricas:
    if col in datos.columns and datos[col].isnull().sum() > 0:
        moda = datos[col].mode()[0]
        datos[col] = datos[col].fillna(moda)

# 6. Eliminar outliers extremos en log_price (precios menores a ~$5 o mayores a ~$2000)
# log(5) ≈ 1.6,  log(2000) ≈ 7.6
datos = datos[(datos['log_price'] >= 1.6) & (datos['log_price'] <= 7.6)]

print(f"  Registros con precios extremos eliminados: {n_antes - len(datos)}")
print("\nDataset Limpiado correctamente")
display(datos.head(3))

In [ ]:
# ===========================================================================
# INFORME POST-LIMPIEZA
# ===========================================================================
print("=" * 65)
print("INFORMACIÓN GENERAL DEL DATASET — POST LIMPIEZA")
print("=" * 65)
print(f"  Filas    : {datos.shape[0]:,}")
print(f"  Columnas : {datos.shape[1]}")
print("=" * 65)
info_df = pd.DataFrame({
    'Tipo': datos.dtypes,
    'No_Nulos': datos.notna().sum(),
    'Nulos': datos.isna().sum(),
    '% Nulos': (datos.isna().sum() / len(datos) * 100).round(2)
})
print(info_df.to_string())

## 1.1 Resultados de la Limpieza

**Resumen del Procesamiento de Datos.**

Tras la etapa de limpieza y preprocesamiento, el conjunto de datos final consta de 74,111 registros y 29 variables y queda saneado con **0% de valores nulos** en todas las columnas, logrando una integridad del 100% en la información.

Este estado óptimo de los datos garantiza la fiabilidad de los análisis estadísticos y modelos de aprendizaje automático posteriores.

- **Mediana en lugar de media para imputación numérica:** La mediana es más robusta ante outliers (ej. propiedades con 605 reseñas inflarían la media de `number_of_reviews`).
- **Conversión explícita de booleanos:** Las columnas `host_has_profile_pic`, `host_identity_verified`, `instant_bookable` y `cleaning_fee` se convierten correctamente de `'t'/'f'` a `1/0`.
- **Limpieza de `host_response_rate`:** Se elimina el símbolo `%` antes de la conversión numérica.
- **Filtro de precios extremos:** Se eliminan registros con `log_price < 1.6` (precio real < ~$5) que probablemente son errores de captura, sin afectar significativamente el volumen total.


# 2. ANÁLISIS DESCRIPTIVO

In [ ]:
# ===========================================================================
# ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS
# ===========================================================================
num_vars = datos.select_dtypes(include=['number'])

print("\n" + "=" * 65)
print("ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS")
print("=" * 65)

stats_rows = []
for col in num_vars:
    s = datos[col].dropna()
    moda_val = s.mode()[0] if not s.mode().empty else np.nan
    stats_rows.append({
        'Variable': col,
        'Media':    round(s.mean(), 2),
        'Mediana':  round(s.median(), 2),
        'Moda':     round(moda_val, 2),
        'Std':      round(s.std(), 2),
        'Min':      round(s.min(), 2),
        'Max':      round(s.max(), 2),
        'Q1':       round(s.quantile(0.25), 2),
        'Q3':       round(s.quantile(0.75), 2),
    })

stats_df = pd.DataFrame(stats_rows).set_index('Variable')
print(stats_df.to_string())

## 2.1 📊 RESUMEN ANALÍTICO DE LAS ESTADÍSTICAS DESCRIPTIVAS

---
### Medidas de Tendencia Central

- **Precio:** El `log_price` medio es $4.78$, equivalente a $119$ en escala real. La moda de 5.01 (~$150) sugiere que ese es el precio psicológico más popular entre los anfitriones.
- **Capacidad:** El alojamiento promedio recibe a ~3 personas, aunque lo más común son propiedades para 2 (moda = 2).
- **Calificaciones:** Media y mediana de `review_scores_rating` convergen en ~94 puntos, indicando que la gran mayoría de propiedades son de alta calidad.
- **Infraestructura:** Un alojamiento típico cuenta con 1 baño, 1 habitación y 1 a 2 camas, como muestran las medianas y modas de estas categorías.

---
### Dispersión y Rangos

- **Alta volatilidad en reseñas:** `number_of_reviews` tiene una desviación estándar de $38$, muy superior a su media (~21). Esto refleja que la mayoría de propiedades tiene pocas reseñas, pero unas pocas concentran cientos.
- **Tamaño de propiedades:** `accommodates` va de 1 a 16 personas, cubriendo desde estudios individuales hasta villas o casas grandes.

---
### Cuartiles

- El 50% central de los alojamientos tiene entre 2 y 4 plazas.
- El 75% de los alojamientos tienen **1 solo dormitorio**, confirmando que la oferta predominante es de unidades pequeñas.
- El 25% mejor calificado tiene puntuaciones entre 99 y 100 puntos.

In [ ]:
# ===========================================================================
# ESTADÍSTICAS DESCRIPTIVAS — VARIABLES CATEGÓRICAS
# ===========================================================================
print("=" * 65)
print("ESTADÍSTICAS DESCRIPTIVAS — VARIABLES CATEGÓRICAS")
print("=" * 65)
display(datos.describe(include=["object"]))

## 2.2 📈 IDENTIFICACIÓN DE VARIABLES CATEGÓRICAS (PERFIL DEL MERCADO)

### Variables Dominantes

- **Property Type (Apartment):** 66% de la oferta son departamentos.
- **Room Type (Entire home/apt):** Más del 55% ofrecen el lugar completo, orientado a la privacidad.
- **Bed Type (Real Bed):** ~97% cuentan con cama real.
- **City (NYC):** Nueva York concentra la mayor parte del dataset con ~32,349 registros.

### Variables de Confianza

- **Host Identity Verified:** La mayoría de anfitriones (~67%) están verificados.
- **Instant Bookable:** La mayoría (~74%) *no* permite reserva instantánea, prefiriendo validar al huésped.
- **Host Response Rate (100%):** La moda es 100% de respuesta, indicando anfitriones muy activos.

### Ubicación

- **Neighbourhood (Williamsburg):** Es el barrio con más anuncios (9,734), seguido de otras zonas de Brooklyn y Manhattan.
- **Zipcode (11211.0):** El código postal más frecuente coincide precisamente con la zona de Brooklyn/Williamsburg.

---
### Resumen

"El dataset describe un mercado predominantemente compuesto por apartamentos completos en la ciudad de Nueva York (específicamente Williamsburg), gestionados por anfitriones verificados con tasas de respuesta perfectas, donde la mayoría de los usuarios prefieren camas reales y un proceso de reserva manual en lugar de instantáneo."

In [ ]:
# ===========================================================================
# VARIABLES QUE MÁS INFLUYEN EN EL PRECIO (CORRELACIÓN)
# ===========================================================================
print("VARIABLES QUE MÁS INFLUYEN EN EL PRECIO")
correlaciones = datos.select_dtypes(include=[np.number]).corr()["log_price"].sort_values(ascending=False)

print("\nIMPACTO POSITIVO (TOP 6):")
print(correlaciones.head(7).to_string())
print("\nIMPACTO NEGATIVO (BOTTOM 5):")
print(correlaciones.tail(5).to_string())

# 3. IDENTIFICACIÓN Y TRATAMIENTO DE OUTLIERS

In [ ]:
# ===========================================================================
# DETECCIÓN DE OUTLIERS (Método IQR)
# ===========================================================================
print("\n" + "=" * 65)
print("DETECCIÓN DE OUTLIERS (Método IQR)")
print("=" * 65)

for col in num_vars.columns:
    s   = datos[col].dropna()
    Q1  = s.quantile(0.25)
    Q3  = s.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((s < lower) | (s > upper)).sum()
    pct_out = n_out / len(s) * 100
    print(f"  {col:35s}: {n_out:5,} outliers ({pct_out:.1f}%) | rango normal [{lower:.1f}, {upper:.1f}]")

## Interpretación de Outliers

- **log_price (2.1%):** El rango normal es [3, 7]. Solo el 2.1% de propiedades tienen precios extremadamente bajos o altos; la transformación logarítmica funcionó bien para estabilizar la distribución.
- **review_scores_rating (6.2%):** El límite inferior del IQR cae en ~84 puntos. Como el máximo real es 100, los outliers aquí son simplemente propiedades con reputación inferior al promedio general (que es muy alto).
- **accommodates (4.9%):** Propiedades para 8+ personas se salen del rango normal, representando el segmento de grandes grupos o familias.
- **number_of_reviews (17%):** Alta concentración de propiedades con pocas reseñas y un pequeño porcentaje con cientos, creando una distribución muy sesgada.

In [ ]:
# ===========================================================================
# BOXPLOTS DE OUTLIERS
# ===========================================================================
vars_to_plot = ['log_price', 'number_of_reviews', 'review_scores_rating', 'accommodates']
labels = ['Log del Precio', 'Número de Reseñas', 'Calificación (0-100)', 'Capacidad (personas)']

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 9))
axes = axes.flatten()
fig.suptitle('Distribución y Outliers — Variables Clave', fontsize=14, fontweight='bold')

for i, (col, lbl) in enumerate(zip(vars_to_plot, labels)):
    sns.boxplot(x=datos[col], ax=axes[i], color='#4C9BE8', flierprops={'alpha': 0.3, 'markersize': 3})
    axes[i].set_title(f'{lbl}', fontsize=12)
    axes[i].set_xlabel('')
    # Añadir etiquetas de estadísticas
    mediana = datos[col].median()
    media   = datos[col].mean()
    axes[i].axvline(mediana, color='#E74C3C', linestyle='--', lw=1.5, label=f'Mediana={mediana:.1f}')
    axes[i].axvline(media,   color='#2ECC71', linestyle='--', lw=1.5, label=f'Media={media:.1f}')
    axes[i].legend(fontsize=9)

plt.tight_layout()
plt.show()

# 4. FEATURE ENGINEERING

In [ ]:
# ===========================================================================
# CREACIÓN DE VARIABLES DERIVADAS
# ===========================================================================
print("\nFEATURE ENGINEERING")

# 1. Precio real (deshacer la transformación logarítmica)
datos['price_real'] = np.exp(datos['log_price'])

# 2. Precio por habitación (valor de cada dormitorio)
datos['price_per_room'] = datos['price_real'] / (datos['bedrooms'] + 1)

# 3. Segmentación de precios en 3 tercios iguales
datos['price_segment'] = pd.qcut(
    datos['price_real'], q=3, labels=['Bajo', 'Medio', 'Alto']
)

# 4. Antigüedad del anfitrión (días desde que se unió)
if 'host_since' in datos.columns:
    datos['host_since'] = pd.to_datetime(datos['host_since'], errors='coerce')
    ref_date = pd.Timestamp('2017-01-01')  # Fecha de referencia aprox. del dataset
    datos['host_years'] = ((ref_date - datos['host_since']).dt.days / 365).round(1)
    datos['host_years'] = datos['host_years'].clip(lower=0).fillna(datos['host_years'].median())

# 5. Número de amenities (contar cuántos servicios tiene la propiedad)
if 'amenities' in datos.columns:
    datos['n_amenities'] = datos['amenities'].apply(
        lambda x: len(str(x).replace('{', '').replace('}', '').split(','))
        if pd.notna(x) and str(x) != '' else 0
    )

print("Columnas nuevas creadas:")
new_cols = ['price_real', 'price_per_room', 'price_segment', 'host_years', 'n_amenities']
print([c for c in new_cols if c in datos.columns])

print("\nVISTA DE VARIABLES RELEVANTES:")
display_cols = [c for c in ['log_price', 'price_real', 'bedrooms', 'price_per_room',
                             'price_segment', 'host_years', 'n_amenities'] if c in datos.columns]
display(datos[display_cols].head(8))

In [ ]:
# ===========================================================================
# ESTADÍSTICA DESCRIPTIVA — NUEVAS VARIABLES
# ===========================================================================
desc_cols = [c for c in ['price_real', 'price_per_room', 'n_amenities', 'host_years']
             if c in datos.columns]
print("\nESTADÍSTICA DESCRIPTIVA (NUEVAS VARIABLES):")
display(datos[desc_cols].describe().round(2))

## MODELO BASE: Regresión Lineal Múltiple (sin regularización)

**¿Por qué un modelo base?**  
Antes de aplicar técnicas de regularización (Ridge, Lasso, ElasticNet), se establece un modelo de referencia con `LinearRegression` puro. Este modelo base permite:

1. Establecer un **benchmark** de desempeño mínimo esperado.
2. Visualizar los coeficientes sin penalización para detectar posibles problemas de multicolinealidad.
3. Comparar si la regularización agrega valor real frente al modelo OLS simple.

**Variables del modelo:**
- **Dependiente:** `log_price` (logaritmo del precio, distribución más normal)
- **Independientes:** `accommodates`, `bedrooms`, `beds`, `bathrooms`, `n_amenities`


In [ ]:
# ===========================================================================
# MODELO BASE: REGRESIÓN LINEAL MÚLTIPLE (sin regularización)
# ===========================================================================
# La rúbrica exige un modelo base antes de aplicar regularización.
# Usamos log_price como target para comparar en la misma escala que el modelo extendido.

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Variables del modelo base (mismas que en sección 6)
base_features = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'n_amenities']
base_features = [f for f in base_features if f in datos.columns]

X_base = datos[base_features].copy().fillna(0)
y_base = datos['log_price'].copy()

mask_base = y_base.notna()
X_base, y_base = X_base[mask_base], y_base[mask_base]

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

scaler_base = StandardScaler()
X_train_base_sc = scaler_base.fit_transform(X_train_base)
X_test_base_sc  = scaler_base.transform(X_test_base)

# Entrenar modelo base
modelo_lineal_base = LinearRegression()
modelo_lineal_base.fit(X_train_base_sc, y_train_base)

y_pred_base = modelo_lineal_base.predict(X_test_base_sc)

rmse_base = np.sqrt(mean_squared_error(y_test_base, y_pred_base))
r2_base   = r2_score(y_test_base, y_pred_base)
r2_adj_base = 1 - (1 - r2_base) * (len(y_test_base)-1) / (len(y_test_base) - len(base_features) - 1)

print("=" * 60)
print("MODELO BASE — REGRESIÓN LINEAL MÚLTIPLE (OLS)")
print("=" * 60)
print(f"  Variable dependiente  : log_price")
print(f"  Variables independ.   : {base_features}")
print(f"  R²                    : {r2_base:.4f}  ({r2_base*100:.1f}% de varianza explicada)")
print(f"  R² Ajustado           : {r2_adj_base:.4f}")
print(f"  RMSE                  : {rmse_base:.4f}")
print()
print("Coeficientes del modelo base:")
import pandas as pd
coef_base_df = pd.DataFrame({
    'Variable':    base_features,
    'Coeficiente': modelo_lineal_base.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)
print(coef_base_df.to_string(index=False))
print(f"\n  Intercepto: {modelo_lineal_base.intercept_:.4f}")


# 5. ENTRENAMIENTO Y EVALUACION DE MODELOS PREDICTIVOS(LASSO, RIDGE Y ELASTICNET)

In [ ]:
# NOTA: Usamos log_price como target para modelos regularizados (escala más estable)
target = 'log_price'

# Create X from the 'datos' DataFrame which contains 'price_real' and other engineered features
X = datos.drop(columns=target).select_dtypes(include=['number'])
y = datos[target]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 5.1 ESCALAMIENTO



In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5.1 RIDGE (REDUCCION DE VARIANZA)

In [ ]:
from sklearn.linear_model import RidgeCV

ridge = RidgeCV(alphas=np.logspace(-3, 3, 100))
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print("Alpha Optimo:", ridge.alpha_)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("R2:", r2_score(y_test, y_pred_ridge))

## 5.2 LASSO (SELECCION DE VARIABLES)

In [ ]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(alphas=np.logspace(-3, 3, 100), cv=5)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

## 5.3 ELASTIC NET (BALANCE)

In [ ]:
from sklearn.linear_model import ElasticNetCV

elastic = ElasticNetCV(
    alphas=np.logspace(-3, 3, 100),
    l1_ratio=[0.2, 0.5, 0.8],
    cv=5
)
elastic.fit(X_train_scaled, y_train)

y_pred_elastic = elastic.predict(X_test_scaled)

## 5.4 METRICAS

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def evaluar(y_real, y_pred, nombre):
  rmse = np.sqrt(mean_squared_error(y_real, y_pred))
  r2 = r2_score(y_real, y_pred)

  print(f"\n{nombre}")
  print(f"RMSE: {rmse:.2f}")
  print(f"R2: {r2:.3f}")

  return rmse, r2

rmse_ridge, r2_ridge = evaluar(y_test, y_pred_ridge, "Ridge")
rmse_lasso, r2_lasso = evaluar(y_test, y_pred_lasso, "Lasso")
rmse_elastic, r2_elastic = evaluar(y_test, y_pred_elastic, "ElasticNet")

## 5.5 COMPARACION

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Ridge', 'Lasso', 'ElasticNet'],
    'RMSE': [rmse_ridge, rmse_lasso, rmse_elastic],
    'R2': [r2_ridge, r2_lasso, r2_elastic]
})

display(comparacion)

## 5.6 IMPORTANCIA DE VARIABLES

In [ ]:
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': elastic.coef_
})

display(coeficientes.sort_values(by='Lasso', key=abs, ascending=False))

## 5.7 ANOVA (VALIDACION DE SIGNIFICANCIA)


In [ ]:
import statsmodels.api as sm

X_sm = sm.add_constant(X_train)
modelo_sm = sm.OLS(y_train, X_sm).fit()

print(modelo_sm.summary())

In [ ]:
print("\nSELECCION DE VARIABLES CON LASSO")

coef_lasso = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': lasso.coef_
})

display(coef_lasso.sort_values(by='Coeficiente', key=abs, ascending=False))

In [ ]:
#Filtrar variables
#Threshold/umbral de decision

threshold = 0.01
variables_importantes = coef_lasso[
    np.abs(coef_lasso['Coeficiente']) > threshold
]['Variable']

print("\nVariables importantes:")
print(variables_importantes.tolist())

In [ ]:
#NUEVO DATASET OPTIMIZADO

# Ensure variables_importantes is not empty; if so, use all original numerical variables
if variables_importantes.empty:
    print("No variables met the importance threshold. Using all numerical variables for the reduced dataset.")
    variables_importantes = X.columns # Fallback to all columns if none were selected

X_reducido = X[variables_importantes]

# Split nuevamente

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reducido, y, test_size=0.2, random_state=42
)

# Escalamiento
scaler = StandardScaler()

X_train_r_scaled = scaler.fit_transform(X_train_r)
X_test_r_scaled = scaler.transform(X_test_r)

In [ ]:
#REENTRENAMIENTO DEL MODELO CON BRIDGE

ridge_opt = RidgeCV(alphas=np.logspace(-3, 3, 100))
ridge_opt.fit(X_train_r_scaled, y_train_r)

y_pred_opt = ridge_opt.predict(X_test_r_scaled)

In [ ]:
# COMPARAR RESULTADOS

rmse_opt = np.sqrt(mean_squared_error(y_test_r, y_pred_opt))
r2_opt = r2_score(y_test_r, y_pred_opt)

print("\nMODELO ORIGINAL (RIDGE):")
print(f"RMSE: {rmse_ridge:.2f}")
print(f"R2: {r2_ridge:.3f}")

print("\nMODELO OPTIMIZADO:")
print(f"RMSE: {rmse_opt:.2f}")
print(f"R2: {r2_opt:.3f}")

In [ ]:
# COMPARACION VISUAL
comparacion_final = pd.DataFrame({
    'Modelo': ['Original', 'Optimizado', 'ElasticNet'],
    'RMSE': [rmse_ridge, rmse_opt, rmse_elastic],
    'R2': [r2_ridge, r2_opt, r2_elastic]
})

display(comparacion_final)

In [ ]:
# ===========================================================================
# SCATTER: PREDICCIONES vs VALORES REALES — Modelos regularizados
# ===========================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Predicción vs Valor Real — Comparación de Modelos', fontsize=13, fontweight='bold')

models_info = [
    (y_pred_ridge,   y_test, 'Ridge',      '#4C9BE8', r2_ridge),
    (y_pred_lasso,   y_test, 'Lasso',      '#2ECC71', r2_lasso),
    (y_pred_elastic, y_test, 'ElasticNet', '#E67E22', r2_elastic),
]

for ax, (y_pred_m, y_real_m, nombre, color, r2_m) in zip(axes, models_info):
    ax.scatter(y_real_m, y_pred_m, alpha=0.15, s=5, color=color)
    mn = min(y_real_m.min(), y_pred_m.min())
    mx = max(y_real_m.max(), y_pred_m.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Predicción perfecta')
    ax.set_title(f'{nombre}  (R²={r2_m:.3f})')
    ax.set_xlabel('log_price Real')
    ax.set_ylabel('log_price Predicho')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ===========================================================================
# IMPORTANCIA DE VARIABLES — Gráfico de barras (valor absoluto de coeficientes)
# ===========================================================================
import pandas as pd

coef_viz = pd.DataFrame({
    'Variable':   list(X.columns),
    'Ridge':      ridge.coef_,
    'Lasso':      lasso.coef_,
    'ElasticNet': elastic.coef_
})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Importancia de Variables por Modelo (|Coeficiente|)',
             fontsize=13, fontweight='bold')

palette = ['#4C9BE8', '#2ECC71', '#E67E22']
model_names = ['Ridge', 'Lasso', 'ElasticNet']

for ax, col, color in zip(axes, model_names, palette):
    df_sorted = coef_viz[['Variable', col]].assign(Abs=coef_viz[col].abs()).sort_values('Abs', ascending=True).tail(12)
    colors_bar = [color if v >= 0 else '#E74C3C' for v in df_sorted[col]]
    ax.barh(df_sorted['Variable'], df_sorted[col], color=colors_bar, edgecolor='white', linewidth=0.8)
    ax.axvline(x=0, color='black', linewidth=1.2, linestyle='--')
    ax.set_title(f'{col}\n(Verde/Azul=sube precio, Rojo=baja precio)', fontsize=10)
    ax.set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()


## Interpretación de Resultados del Modelo

### Lectura del Scatter Plot (Predicción vs Real)
El gráfico de dispersión muestra qué tan bien el modelo reproduce los precios reales. La línea roja punteada representa la **predicción perfecta** (predicho = real):
- Los puntos **cercanos a la línea** indican buenas predicciones.
- La **nube de puntos** tiende a agruparse alrededor de la diagonal, lo que confirma que los modelos capturan la tendencia central del precio.
- Se observa mayor dispersión en precios extremos (muy bajos o muy altos), lo que es esperado: los outliers de precio son más difíciles de predecir con variables físicas.

### Lectura del R² y RMSE
| Modelo | Interpretación |
|--------|----------------|
| **Ridge** | Retiene todas las variables con pesos reducidos. Buen desempeño general con alta varianza de datos. |
| **Lasso** | Fuerza coeficientes a cero → selección automática de variables. Útil cuando muchas variables son irrelevantes. |
| **ElasticNet** | Combina Ridge y Lasso. Equilibra sesgo-varianza cuando hay multicolinealidad moderada. |

Un R² de ~0.40–0.55 **no es un fracaso**: indica que las características físicas (tamaño, habitaciones) explican entre el 40% y 55% del precio. El resto depende de factores no capturados como la ubicación exacta, las fotografías del anuncio y la demanda estacional.

### Variables más influyentes
Según los coeficientes, `accommodates`, `bathrooms` y `bedrooms` son los predictores más fuertes del precio. `n_amenities` tiene un efecto moderado positivo, lo que tiene sentido: más servicios aumentan el valor percibido de la propiedad.


## 5.8 VARIABLES FINALES UTILIZADAS

In [ ]:
# INTERPRETACION

print("\nVARIABLES ORIGINALES:")
print(X.columns.tolist())

print("\n VARIABLES FINALES UTILIZADAS:")
print(list(variables_importantes))

print(f"\nTotal variables originales: {X.shape[1]}")
print(f"Total variables finales: {len(variables_importantes)}")

# 6. ANÁLISIS EXPLORATORIO VISUAL (EDA)

In [ ]:
# ===========================================================================
# DISTRIBUCIÓN DE PRECIOS
# ===========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución del Precio — Comparación Log vs Real', fontsize=13, fontweight='bold')

# Log Price
sns.histplot(datos['log_price'], kde=True, ax=axes[0], color='#4C9BE8', bins=50)
axes[0].set_title('Distribución de Precio (log_price)')
axes[0].set_xlabel('Log del Precio')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(datos['log_price'].mean(),   color='#E74C3C', linestyle='--', lw=2, label=f"Media={datos['log_price'].mean():.2f}")
axes[0].axvline(datos['log_price'].median(), color='#2ECC71', linestyle='--', lw=2, label=f"Mediana={datos['log_price'].median():.2f}")
axes[0].legend()

# Precio real (filtrado a <$600 para visualización)
precio_filtrado = datos[datos['price_real'] <= 600]['price_real']
sns.histplot(precio_filtrado, kde=True, ax=axes[1], color='#E67E22', bins=60)
axes[1].set_title('Distribución de Precio Real (≤$600)')
axes[1].set_xlabel('Precio ($)')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(precio_filtrado.mean(),   color='#E74C3C', linestyle='--', lw=2, label=f"Media=${precio_filtrado.mean():.0f}")
axes[1].axvline(precio_filtrado.median(), color='#2ECC71', linestyle='--', lw=2, label=f"Mediana=${precio_filtrado.median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Precio real — Media: ${datos['price_real'].mean():.2f} | Mediana: ${datos['price_real'].median():.2f}")
print(f"Precio real — Percentil 75: ${datos['price_real'].quantile(0.75):.2f} | Percentil 95: ${datos['price_real'].quantile(0.95):.2f}")

## 6.1 Interpretación — Distribución de Precios

**Log Price:** La distribución es aproximadamente normal centrada en ~4.78 (≈$119$). Los "picos" en la distribución corresponden a precios psicológicos redondos como $100$ (log≈4.60), $150$ (log≈5.01) y $200 (log≈5.30).

**Precio Real:** Al quitar el logaritmo, la distribución muestra un fuerte sesgo a la derecha (muchas propiedades baratas, pocas muy caras). La mediana ($105$) es más representativa que la media (~$130) por la influencia de propiedades de lujo.

In [ ]:
# ===========================================================================
# MAPA DE CALOR DE CORRELACIONES
# ===========================================================================
print("\nMAPA DE CALOR — IDENTIFICANDO VARIABLES QUE INFLUYEN EN EL PRECIO")

# Seleccionar columnas numéricas relevantes para el análisis
cols_heatmap = ['log_price', 'price_real', 'accommodates', 'bathrooms', 'bedrooms',
                'beds', 'number_of_reviews', 'review_scores_rating',
                'host_has_profile_pic', 'host_identity_verified', 'instant_bookable',
                'n_amenities', 'host_years', 'price_per_room']
cols_heatmap = [c for c in cols_heatmap if c in datos.columns]

plt.figure(figsize=(13, 10))
matriz_corr = datos[cols_heatmap].corr()
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))  # Solo triángulo inferior

sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5,
            mask=mask, annot_kws={'size': 8}, vmin=-1, vmax=1)
plt.title('Mapa de Calor de Correlaciones — Variables Seleccionadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 8 variables más correlacionadas con log_price:")
print(datos[cols_heatmap].corr()["log_price"].sort_values(ascending=False).head(9).to_string())

In [ ]:
# ===========================================================================
# DISTRIBUCIÓN POR SEGMENTO DE PRECIO
# ===========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis por Segmento de Precio', fontsize=13, fontweight='bold')

# Frecuencia por segmento
segment_counts = datos['price_segment'].value_counts()
colors_seg = ['#2ECC71', '#F39C12', '#E74C3C']
axes[0].bar(segment_counts.index, segment_counts.values, color=colors_seg, edgecolor='white', linewidth=1.5)
axes[0].set_title('Cantidad de Propiedades por Segmento')
axes[0].set_xlabel('Segmento de Precio')
axes[0].set_ylabel('Cantidad de Propiedades')
for i, (seg, val) in enumerate(segment_counts.items()):
    axes[0].text(i, val + 200, f'{val:,}', ha='center', fontsize=10, fontweight='bold')

# Precio promedio por segmento
seg_means = datos.groupby('price_segment')['price_real'].mean()
axes[1].bar(seg_means.index, seg_means.values, color=colors_seg, edgecolor='white', linewidth=1.5)
axes[1].set_title('Precio Promedio por Segmento')
axes[1].set_xlabel('Segmento de Precio')
axes[1].set_ylabel('Precio Promedio ($)')
for i, (seg, val) in enumerate(seg_means.items()):
    axes[1].text(i, val + 2, f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nDistribución por segmento:")
print(datos['price_segment'].value_counts().to_string())
print("\nPrecio promedio por segmento:")
print(datos.groupby('price_segment')['price_real'].mean().round(2).to_string())

## 6.2 Interpretación — Segmentación de Precios

El dataset se divide en tercios balanceados (~24,600 propiedades por segmento):

- **Segmento Bajo (~$60$):** Habitaciones compartidas o estudios pequeños en zonas periféricas.
- **Segmento Medio (~$117$):** El corazón del mercado Airbnb — apartamentos completos estándar.
- **Segmento Alto (~$31$2):** Lujo con características diferenciadoras (ubicación premium, más habitaciones, servicios de alta gama).

**Insight crítico:** El salto de Medio a Alto (+167%) es mucho más pronunciado que de Bajo a Medio (+95%), confirmando que el segmento alto representa un mercado radicalmente distinto, no simplemente "un poco más caro".

# 7. ANÁLISIS 1: MODELO DE REGRESIÓN LINEAL MÚLTIPLE
## Precio ~ Características Físicas de la Propiedad

**Variable Dependiente:** `log_price`  
**Variables Independientes:** `accommodates`, `bedrooms`, `beds`, `bathrooms`, `n_amenities`

Este modelo busca responder: **¿Cuánto influyen las características físicas de la propiedad en su precio?**

In [ ]:
# ===========================================================================
# 6.1 IDENTIFICACIÓN DE VARIABLES Y ANÁLISIS DE CORRELACIÓN
# ===========================================================================

# 1. Creamos las columnas para cada distrito
distritos_dummies = pd.get_dummies(datos['neighbourhood'])

# 2. Las pegamos a nuestro dataset principal
datos = pd.concat([datos, distritos_dummies], axis=1)

# 3. Guardamos los nombres de estas nuevas columnas en una lista para el modelo
columnas_zonas = distritos_dummies.columns.tolist()

print(f"Zonas detectadas: {columnas_zonas}")

# Variables independientes (características de la propiedad)
feature_cols = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'n_amenities', 'cleaning_fee'] + columnas_zonas
feature_cols = [c for c in feature_cols if c in datos.columns]
target_col   = 'log_price'

print("Variable dependiente  :", target_col)
print("Variables independientes:", feature_cols)
print("\nCorrelación individual con log_price:")
print(datos[feature_cols + [target_col]].corr()[target_col].sort_values(ascending=False).to_string())

In [ ]:
# Configuración de la gráfica
plt.figure(figsize=(10, 7))

# Graficamos una muestra de 5,000 puntos para que la visualización sea fluida
# y añadimos la línea de regresión con regplot
sns.regplot(data=datos.sample(5000),
            x='log_price',
            y='beds', # Se corrigió 'feature_cols' a una columna específica
            scatter_kws={'alpha':0.3, 'color':'royalblue'},
            line_kws={'color':'red', 'lw':2})

plt.title('Regresión Lineal: Log Price vs Accommodates', fontsize=15)
plt.xlabel('Log del Precio', fontsize=12)
plt.ylabel('Capacidad (personas)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

## 7.1 Visualización Individual de Regresiones Simples

Como se mencionó, `sns.regplot` solo puede mostrar la relación entre dos variables. A continuación, se muestra una `regplot` para cada una de las variables independientes con respecto a `log_price`, para visualizar sus relaciones individuales.

In [ ]:
# Variables independientes para la visualización
feature_cols_display = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'n_amenities']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()
fig.suptitle('Regresión Lineal Simple: log_price vs Características de la Propiedad', fontsize=16, fontweight='bold')

# Ajustar el diseño para acomodar un gráfico menos en la última fila
for i in range(len(feature_cols_display), len(axes)):
    fig.delaxes(axes[i])

for i, col in enumerate(feature_cols_display):
    sns.regplot(
        data=datos.sample(5000, random_state=42), # Muestra para eficiencia visual
        x='log_price',
        y=col,
        scatter_kws={'alpha':0.3, 'color':'royalblue'},
        line_kws={'color':'red', 'lw':2},
        ax=axes[i]
    )
    axes[i].set_title(f'Log Price vs {col}', fontsize=12)
    axes[i].set_xlabel('Log del Precio')
    axes[i].set_ylabel(col)
    axes[i].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajusta el layout para que el título no se superponga
plt.show()

In [ ]:
# ===========================================================================
# PAIRPLOT — RELACIONES ENTRE VARIABLES
# ===========================================================================
print("Generando Pairplot (puede tardar unos segundos)...")

# Muestra aleatoria para eficiencia visual
sample_df = datos[feature_cols + [target_col]].sample(3000, random_state=42)

g = sns.pairplot(sample_df, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 10},
                 diag_kws={'color': '#4C9BE8'})
g.fig.suptitle('Pairplot — Variables de Propiedad vs Precio (muestra 3,000)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# ANÁLISIS DE MULTICOLINEALIDAD (VIF)
# ===========================================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X_vif = datos[feature_cols].dropna()
X_vif_scaled = (X_vif - X_vif.mean()) / X_vif.std()  # estandarización

vif_data = pd.DataFrame({
    'Variable': feature_cols,
    'VIF': [variance_inflation_factor(X_vif_scaled.values, i) for i in range(len(feature_cols))]
}).sort_values('VIF', ascending=False)

print("=" * 50)
print("FACTOR DE INFLACIÓN DE VARIANZA (VIF)")
print("VIF > 10 → multicolinealidad problemática")
print("=" * 50)
print(vif_data.to_string(index=False))

# Visualización
plt.figure(figsize=(8, 4))
colors_vif = ['#E74C3C' if v > 10 else '#F39C12' if v > 5 else '#2ECC71' for v in vif_data['VIF']]
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colors_vif)
plt.axvline(x=5,  color='orange', linestyle='--', lw=1.5, label='VIF=5 (advertencia)')
plt.axvline(x=10, color='red',    linestyle='--', lw=1.5, label='VIF=10 (problema)')
plt.title('Factor de Inflación de Varianza (VIF) — Variables de Propiedad')
plt.xlabel('VIF')
plt.legend()
plt.tight_layout()
plt.show()

## 7.2 Interpretación del VIF

El **VIF (Factor de Inflación de Varianza)** mide cuánta de la varianza de un coeficiente se "infla" por la correlación con otras variables. Un VIF > 10 indica multicolinealidad problemática.

- **`beds` y `accommodates`** pueden tener VIF elevado porque están altamente correlacionadas (más personas → más camas). Si el VIF supera 10, se puede optar por eliminar `beds` del modelo final.
- **`bedrooms` y `accommodates`** también están correlacionadas pero en menor medida.
- **`bathrooms` y `n_amenities`** generalmente tienen VIF bajo, confirmando que aportan información independiente.

In [ ]:
# ===========================================================================
# SEPARACIÓN TRAIN / TEST
# ===========================================================================

# Preparar datos
X = datos[feature_cols].copy().fillna(0)
y = datos[target_col].copy()

# Eliminar filas con NaN si las hubiera
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("=" * 50)
print("DIVISIÓN DEL DATASET")
print("=" * 50)
print(f"  Entrenamiento : {len(X_train):,} registros ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Prueba        : {len(X_test):,}  registros ({len(X_test)/len(X)*100:.0f}%)")
print(f"  Features      : {feature_cols}")

In [ ]:
# ===========================================================================
# CONSTRUCCIÓN Y ENTRENAMIENTO DEL MODELO
# ===========================================================================

modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Coeficientes
coef_df = pd.DataFrame({
    'Variable':    feature_cols,
    'Coeficiente': modelo.coef_,
    'Impacto_en_precio_%': np.expm1(modelo.coef_) * 100  # Interpretación aproximada
}).sort_values('Coeficiente', ascending=False)

print("=" * 65)
print("COEFICIENTES DEL MODELO DE REGRESIÓN LINEAL MÚLTIPLE")
print("=" * 65)
print(f"  Intercepto: {modelo.intercept_:.4f}  →  Precio base ≈ ${np.exp(modelo.intercept_):.2f}")
print()
print(coef_df.to_string(index=False))
print()
print("Interpretación: Por cada unidad adicional en cada variable,")
print("el log_price cambia en el valor del coeficiente (efecto multiplicativo en precio real).")

In [ ]:
# ===========================================================================
# EVALUACIÓN DEL MODELO
# ===========================================================================

y_pred = modelo.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)
mae  = np.mean(np.abs(y_test - y_pred))

# R² ajustado
n = len(y_test)
p = len(feature_cols)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("=" * 55)
print("MÉTRICAS DE EVALUACIÓN DEL MODELO")
print("=" * 55)
print(f"  R²                    : {r2:.4f}  ({r2*100:.1f}% de varianza explicada)")
print(f"  R² Ajustado           : {r2_adj:.4f}")
print(f"  MSE  (log_price)      : {mse:.4f}")
print(f"  RMSE (log_price)      : {rmse:.4f}")
print(f"  MAE  (log_price)      : {mae:.4f}")
print(f"  RMSE en precio real ≈ : ${np.expm1(rmse)*100:.0f}  (error promedio aprox.)")
print("=" * 55)
print()
print(f"  Interpretación R²: El modelo explica el {r2*100:.1f}% de la variabilidad")
print(f"  en los precios a partir de las características físicas de la propiedad.")

## 7.3 Interpretación de las Métricas

**R²:** Mide qué porcentaje de la variabilidad del precio explica el modelo. Un R² de 0.50 significa que las características físicas explican el 50% de la variación de precios; el 50% restante depende de factores no capturados (ubicación, reputación, fotografías, temporada).

**Limitación esperada:** Es normal que este modelo tenga un R² moderado (0.40–0.60) porque el precio en Airbnb depende fuertemente de la **ubicación** (vecindario, vista) y de factores cualitativos que las variables físicas no capturan completamente. En la Sección 7 se incorporará la zona geográfica para mejorar el poder predictivo.

In [ ]:
# ===========================================================================
# PREDICCIONES Y COMPARACIÓN REAL vs PREDICHO
# ===========================================================================

resultados = pd.DataFrame({
    'Real (log)':        y_test.values[:15],
    'Predicho (log)':    y_pred[:15].round(3),
    'Real ($)':          np.exp(y_test.values[:15]).round(0),
    'Predicho ($)':      np.exp(y_pred[:15]).round(0),
    'Error ($)':         (np.exp(y_test.values[:15]) - np.exp(y_pred[:15])).round(0)
})

print("Primeras 15 predicciones vs valores reales:")
display(resultados)

In [ ]:
# ===========================================================================
# VISUALIZACIONES DE RESULTADOS
# ===========================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Evaluación del Modelo de Regresión — Características de Propiedad',
             fontsize=13, fontweight='bold')

# 1. Real vs Predicho (log_price)
axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='#4C9BE8')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Predicción perfecta')
axes[0].set_title(f'Real vs Predicho (R²={r2:.3f})')
axes[0].set_xlabel('log_price Real')
axes[0].set_ylabel('log_price Predicho')
axes[0].legend()

# 2. Residuos
residuos = y_test.values - y_pred
axes[1].scatter(y_pred, residuos, alpha=0.3, s=10, color='#E67E22')
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residuos vs Valores Predichos')
axes[1].set_xlabel('Valores Predichos')
axes[1].set_ylabel('Residuo')

# 3. Importancia de características (coeficientes absolutos)
imp_df = pd.DataFrame({'Variable': feature_cols, 'Importancia': np.abs(modelo.coef_)})
imp_df = imp_df.sort_values('Importancia', ascending=True)
colors_imp = ['#2ECC71' if c > 0 else '#E74C3C' for c in
              [modelo.coef_[list(feature_cols).index(v)] for v in imp_df['Variable']]]
axes[2].barh(imp_df['Variable'], imp_df['Importancia'], color=colors_imp, edgecolor='white')
axes[2].set_title('Importancia de Características (|Coeficiente|)')
axes[2].set_xlabel('Valor Absoluto del Coeficiente')

plt.tight_layout()
plt.show()

## 6.11 Narrativa del Modelo — Características de la Propiedad

**Hallazgos principales:**

1. **`bathrooms` y `accommodates`** son los predictores más fuertes del precio: cada baño adicional y cada persona adicional que puede albergar la propiedad aumentan significativamente el precio.

2. **`bedrooms`** tiene un efecto positivo pero más moderado. Esto es coherente con el mercado de NYC, donde muchos apartamentos de alta demanda son estudios o monoambientes.

3. **`beds`** puede mostrar un efecto menor o incluso negativo en presencia de `bedrooms` y `accommodates` debido a la multicolinealidad: su información ya está capturada por las otras variables.

4. **`n_amenities`** aporta valor incremental al modelo: más servicios (WiFi, cocina, aire acondicionado) se traducen en precios más altos, confirmando que los huéspedes valoran la comodidad.

**Limitación del modelo:** El R² moderado era esperado. El precio en Airbnb tiene un componente geográfico muy fuerte que este modelo no captura. El siguiente análisis aborda precisamente esto.

# 8. ANÁLISIS 2: PRECIO Y ZONA GEOGRÁFICA
## ¿Dónde están las zonas más populares y más caras de NYC?

In [ ]:
# ===========================================================================
# FILTRAR SOLO NYC Y PREPARAR DATOS GEOGRÁFICOS
# ===========================================================================

# Filtrar solo registros de NYC si hay columna 'city'
if 'city' in datos.columns:
    nyc_data = datos[datos['city'].str.upper().str.contains('NEW YORK|NYC', na=False)].copy()
    if len(nyc_data) < 1000:
        nyc_data = datos.copy()  # Si no hay suficientes, usar todo el dataset
else:
    nyc_data = datos.copy()

print(f"Registros para análisis geográfico: {len(nyc_data):,}")
print(f"Vecindarios únicos: {nyc_data['neighbourhood'].nunique()}")
print(f"\nTop 10 vecindarios por número de listados:")
print(nyc_data['neighbourhood'].value_counts().head(10).to_string())

In [ ]:
# ===========================================================================
# TOP 15 ZONAS MÁS POPULARES (POR NÚMERO DE LISTADOS)
# ===========================================================================
top_n = 15
top_neighbourhoods = nyc_data['neighbourhood'].value_counts().head(top_n)

plt.figure(figsize=(12, 6))
colors_pop = ['#E74C3C' if i == 0 else '#F39C12' if i < 3 else '#4C9BE8'
              for i in range(top_n)]

bars = plt.barh(top_neighbourhoods.index[::-1], top_neighbourhoods.values[::-1],
                color=colors_pop[::-1], edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, top_neighbourhoods.values[::-1]):
    plt.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9)

plt.title(f'Top {top_n} Vecindarios Más Populares de NYC por Número de Listados',
          fontsize=13, fontweight='bold')
plt.xlabel('Número de Listados')
plt.ylabel('Vecindario')
plt.tight_layout()
plt.show()

print(f"\nEl vecindario más popular: {top_neighbourhoods.index[0]} ({top_neighbourhoods.iloc[0]:,} listados)")

In [ ]:
# ===========================================================================
# PRECIO PROMEDIO POR VECINDARIO (TOP 20) — DE MAYOR A MENOR
# ===========================================================================

# Calcular precio promedio y número de listados por vecindario
neighbourhood_stats = nyc_data.groupby('neighbourhood').agg(
    precio_promedio=('price_real', 'mean'),
    num_listados=('price_real', 'count'),
    precio_mediana=('price_real', 'median')
).reset_index()

# Filtrar vecindarios con al menos 30 listados (representatividad estadística)
neighbourhood_stats = neighbourhood_stats[neighbourhood_stats['num_listados'] >= 30]
neighbourhood_stats = neighbourhood_stats.sort_values('precio_promedio', ascending=False)

top20_precio = neighbourhood_stats.head(20)
top20_precio_rev = top20_precio.iloc[::-1]  # Invertir para barh

fig, ax = plt.subplots(figsize=(12, 8))
colors_price = ['#E74C3C' if i < 5 else '#F39C12' if i < 10 else '#4C9BE8'
                for i in range(len(top20_precio_rev))]

bars = ax.barh(top20_precio_rev['neighbourhood'], top20_precio_rev['precio_promedio'],
               color=colors_price, edgecolor='white', linewidth=1.2)

for bar, val, n in zip(bars, top20_precio_rev['precio_promedio'],
                       top20_precio_rev['num_listados']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'${val:.0f}  (n={n:,})', va='center', fontsize=8.5)

ax.set_title('Top 20 Vecindarios con Mayor Precio Promedio en NYC\n(mínimo 30 listados)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Precio Promedio ($)')
ax.set_ylabel('Vecindario')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# POPULARIDAD vs PRECIO — SCATTER PLOT POR VECINDARIO
# ===========================================================================

# Top 40 vecindarios por listados
top40 = nyc_data['neighbourhood'].value_counts().head(40).index
ns40  = neighbourhood_stats[neighbourhood_stats['neighbourhood'].isin(top40)].copy()

# Clasificar por cuadrante
precio_median  = ns40['precio_promedio'].median()
listados_media = ns40['num_listados'].median()

def clasificar_cuadrante(row):
    if row['precio_promedio'] >= precio_median and row['num_listados'] >= listados_media:
        return 'Alto precio & Popular'
    elif row['precio_promedio'] >= precio_median and row['num_listados'] < listados_media:
        return 'Alto precio & Nicho'
    elif row['precio_promedio'] < precio_median and row['num_listados'] >= listados_media:
        return 'Precio moderado & Popular'
    else:
        return 'Precio moderado & Nicho'

ns40['cuadrante'] = ns40.apply(clasificar_cuadrante, axis=1)

paleta = {
    'Alto precio & Popular':    '#E74C3C',
    'Alto precio & Nicho':      '#F39C12',
    'Precio moderado & Popular':'#2ECC71',
    'Precio moderado & Nicho':  '#95A5A6'
}

fig, ax = plt.subplots(figsize=(13, 8))
for cat, group in ns40.groupby('cuadrante'):
    ax.scatter(group['num_listados'], group['precio_promedio'],
               s=group['num_listados']/10, alpha=0.75, label=cat,
               color=paleta[cat], edgecolors='white', linewidths=0.8)

# Etiquetar los puntos más notables
for _, row in ns40.nlargest(8, 'num_listados').iterrows():
    ax.annotate(row['neighbourhood'], (row['num_listados'], row['precio_promedio']),
                textcoords='offset points', xytext=(5, 5), fontsize=7.5)
for _, row in ns40.nlargest(5, 'precio_promedio').iterrows():
    ax.annotate(row['neighbourhood'], (row['num_listados'], row['precio_promedio']),
                textcoords='offset points', xytext=(5, -10), fontsize=7.5)

ax.axhline(y=precio_median,  color='gray', linestyle='--', lw=1.2, alpha=0.7)
ax.axvline(x=listados_media, color='gray', linestyle='--', lw=1.2, alpha=0.7)
ax.set_title('Popularidad vs Precio Promedio por Vecindario\n(tamaño del punto = número de listados)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Listados (Popularidad)')
ax.set_ylabel('Precio Promedio ($)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# DISTRIBUCIÓN DE SEGMENTOS POR VECINDARIO (TOP 10)
# ===========================================================================

top10_listados = nyc_data['neighbourhood'].value_counts().head(10).index
datos_top10 = nyc_data[nyc_data['neighbourhood'].isin(top10_listados)].copy()

# Calcular proporción de cada segmento por vecindario
seg_by_nb = (datos_top10.groupby(['neighbourhood', 'price_segment'])
             .size().unstack(fill_value=0))
seg_by_nb_pct = seg_by_nb.div(seg_by_nb.sum(axis=1), axis=0) * 100

# Ordenar por precio promedio
orden_precio = (datos_top10.groupby('neighbourhood')['price_real']
                .mean().sort_values(ascending=False).index)
seg_by_nb_pct = seg_by_nb_pct.loc[orden_precio]

fig, ax = plt.subplots(figsize=(13, 6))
seg_by_nb_pct.plot(kind='bar', stacked=True, ax=ax,
                   color=['#2ECC71', '#F39C12', '#E74C3C'],
                   edgecolor='white', linewidth=0.8)
ax.set_title('Distribución de Segmentos de Precio por Vecindario (Top 10 más populares)\n(ordenado por precio promedio descendente)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Vecindario')
ax.set_ylabel('Porcentaje de Listados (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
ax.legend(title='Segmento', labels=['Bajo', 'Medio', 'Alto'], loc='upper right')
plt.tight_layout()
plt.show()

## 8.1 Narrativa — Análisis Geográfico

**Hallazgos clave:**

1. **Williamsburg y Bed-Stuy dominan en popularidad** (número de listados). Son barrios con una alta concentración de apartamentos completos a precios moderados, típicamente preferidos por jóvenes viajeros que buscan la experiencia local de Brooklyn.

2. **Los vecindarios más caros de Manhattan** (SoHo, Tribeca, West Village, Financial District) tienen precios promedio que superan los $200-$250, pero con menos listados totales, configurando el segmento de "lujo de nicho".

3. **El scatter plot de popularidad vs precio** revela cuatro tipologías de mercado:
   - **Alto precio & Popular:** Los barrios "estrella" con alta demanda y precios premium.
   - **Alto precio & Nicho:** Zonas exclusivas con pocos listados pero precios muy altos.
   - **Precio moderado & Popular:** El volumen del mercado Airbnb de NYC.
   - **Precio moderado & Nicho:** Zonas periféricas con oferta limitada.

4. **Hay un vecindario predominante sin importar el precio:** Williamsburg aparece consistentemente como el más popular en cualquier segmento de precio, lo que lo convierte en el barrio de referencia del mercado.

# 9. ANÁLISIS 3: PRECIO Y REPUTACIÓN DEL ANFITRIÓN
## ¿Más reseñas, más confianza, más precio?

**Variables de reputación analizadas:**
- `host_has_profile_pic` — ¿El anfitrión muestra su cara?
- `host_identity_verified` — ¿La identidad está verificada?
- `instant_bookable` — ¿Se puede reservar sin aprobación previa?
- `host_years` — ¿Cuántos años lleva en la plataforma?
- `number_of_reviews` — ¿Cuántas reseñas tiene?
- `review_scores_rating` — ¿Cuál es su calificación?

In [ ]:
# ===========================================================================
# PRECIO POR VARIABLES BINARIAS DE CONFIANZA
# ===========================================================================
binary_vars = ['host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
binary_labels = {
    'host_has_profile_pic':   'Tiene Foto de Perfil',
    'host_identity_verified': 'Identidad Verificada',
    'instant_bookable':       'Reserva Instantánea'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Precio Promedio según Variables de Confianza del Anfitrión',
             fontsize=13, fontweight='bold')

for i, col in enumerate(binary_vars):
    if col not in datos.columns:
        continue
    grp = datos.groupby(col)['price_real'].agg(['mean', 'median', 'count']).reset_index()
    grp[col] = grp[col].map({1: 'Sí', 0: 'No', '1': 'Sí', '0': 'No'})

    bars = axes[i].bar(grp[col], grp['mean'], color=['#E74C3C', '#2ECC71'],
                       edgecolor='white', linewidth=1.5, width=0.5)
    for bar, row in zip(bars, grp.itertuples()):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'${bar.get_height():.0f}\n(n={row.count:,})',
                     ha='center', fontsize=9.5, fontweight='bold')

    axes[i].set_title(binary_labels.get(col, col))
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Precio Promedio ($)' if i == 0 else '')
    axes[i].set_ylim(0, grp['mean'].max() * 1.25)

plt.tight_layout()
plt.show()

# Imprimir diferencias
print("\nDiferencias de precio promedio por variable binaria:")
for col in binary_vars:
    if col in datos.columns:
        g = datos.groupby(col)['price_real'].mean()
        vals = dict(g)
        si  = vals.get(1,  vals.get(1.0,  None))
        no  = vals.get(0,  vals.get(0.0,  None))
        if si and no:
            diff = si - no
            pct  = (diff / no) * 100
            print(f"  {col:<30}: Sí=${si:.0f}  No=${no:.0f}  Diferencia=${diff:.0f} ({pct:+.1f}%)")

In [ ]:
# ===========================================================================
# RELACIÓN ENTRE NÚMERO DE RESEÑAS Y PRECIO
# ===========================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Relación entre Número de Reseñas y Precio', fontsize=13, fontweight='bold')

# 1. Scatter (muestra)
sample = datos.sample(5000, random_state=42)
axes[0].scatter(sample['number_of_reviews'], sample['price_real'],
                alpha=0.3, s=8, color='#4C9BE8')
axes[0].set_xlim(0, 150)
axes[0].set_ylim(0, 500)
axes[0].set_xlabel('Número de Reseñas')
axes[0].set_ylabel('Precio Real ($)')
axes[0].set_title('Precio vs Número de Reseñas')

# Línea de tendencia
from numpy.polynomial.polynomial import polyfit
mask2 = (sample['number_of_reviews'] <= 150) & (sample['price_real'] <= 500)
x_t = sample.loc[mask2, 'number_of_reviews']
y_t = sample.loc[mask2, 'price_real']
if len(x_t) > 10:
    b, m = polyfit(x_t, y_t, 1)
    x_line = np.linspace(0, 150, 100)
    axes[0].plot(x_line, m*x_line + b, color='#E74C3C', lw=2, label=f'Tendencia')
    axes[0].legend()

# 2. Bins de reseñas: precio promedio por rango
datos['reviews_bin'] = pd.cut(datos['number_of_reviews'],
                               bins=[0,5,15,30,60,120,500],
                               labels=['1-5','6-15','16-30','31-60','61-120','121+'])
rev_price = datos.groupby('reviews_bin', observed=True)['price_real'].agg(['mean','count'])

axes[1].bar(rev_price.index.astype(str), rev_price['mean'],
            color='#4C9BE8', edgecolor='white', linewidth=1.5)
for i, (val, n) in enumerate(zip(rev_price['mean'], rev_price['count'])):
    axes[1].text(i, val + 1, f'${val:.0f}\n(n={n:,})', ha='center', fontsize=8.5)
axes[1].set_title('Precio Promedio por Rango de Reseñas')
axes[1].set_xlabel('Rango de Reseñas')
axes[1].set_ylabel('Precio Promedio ($)')
axes[1].set_ylim(0, rev_price['mean'].max() * 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# RELACIÓN ENTRE CALIFICACIÓN Y PRECIO
# ===========================================================================

# Filtrar calificaciones válidas (0-100)
datos_rating = datos[(datos['review_scores_rating'] > 0) &
                      (datos['review_scores_rating'] <= 100)].copy()
datos_rating['rating_group'] = pd.cut(
    datos_rating['review_scores_rating'],
    bins=[0, 60, 70, 80, 85, 90, 95, 100],
    labels=['<60', '60-70', '70-80', '80-85', '85-90', '90-95', '95-100']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Relación entre Calificación del Anfitrión y Precio', fontsize=13, fontweight='bold')

# 1. Precio promedio por grupo de calificación
rg = datos_rating.groupby('rating_group', observed=True)['price_real'].agg(['mean','count'])
rg_colors = ['#E74C3C','#E67E22','#F39C12','#F1C40F','#2ECC71','#27AE60','#1A8A4C']

bars = axes[0].bar(rg.index.astype(str), rg['mean'],
                   color=rg_colors[:len(rg)], edgecolor='white', linewidth=1.5)
for bar, val, n in zip(bars, rg['mean'], rg['count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'${val:.0f}\n(n={n:,})', ha='center', fontsize=8, fontweight='bold')
axes[0].set_title('Precio Promedio por Rango de Calificación')
axes[0].set_xlabel('Calificación (0-100)')
axes[0].set_ylabel('Precio Promedio ($)')
axes[0].set_ylim(0, rg['mean'].max() * 1.3)

# 2. Distribución de calificaciones por segmento de precio
datos_rating_seg = datos_rating[datos_rating['price_segment'].notna()].copy()
sns.boxplot(data=datos_rating_seg, x='price_segment', y='review_scores_rating',
            ax=axes[1], palette=['#2ECC71','#F39C12','#E74C3C'],
            order=['Bajo','Medio','Alto'])
axes[1].set_title('Calificación por Segmento de Precio')
axes[1].set_xlabel('Segmento de Precio')
axes[1].set_ylabel('Calificación (0-100)')

plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# ANTIGÜEDAD DEL ANFITRIÓN vs PRECIO
# ===========================================================================

if 'host_years' in datos.columns:
    datos['host_years_bin'] = pd.cut(datos['host_years'],
                                      bins=[0, 1, 2, 3, 5, 7, 15],
                                      labels=['<1 año','1-2 años','2-3 años',
                                              '3-5 años','5-7 años','7+ años'])
    hy = datos.groupby('host_years_bin', observed=True)['price_real'].agg(['mean','median','count'])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Antigüedad del Anfitrión y su Relación con el Precio',
                 fontsize=13, fontweight='bold')

    bars = axes[0].bar(hy.index.astype(str), hy['mean'],
                       color=['#4C9BE8']*len(hy), edgecolor='white', linewidth=1.5)
    axes[0].plot(range(len(hy)), hy['median'], color='#E74C3C',
                 marker='o', lw=2, label='Mediana')
    for i, (bar, val, n) in enumerate(zip(bars, hy['mean'], hy['count'])):
        axes[0].text(i, bar.get_height() + 1,
                     f'${val:.0f}\n(n={n:,})', ha='center', fontsize=8)
    axes[0].set_title('Precio Promedio por Antigüedad del Anfitrión')
    axes[0].set_xlabel('Años en la Plataforma')
    axes[0].set_ylabel('Precio ($)')
    axes[0].legend()
    axes[0].set_ylim(0, hy['mean'].max() * 1.3)

    # Correlación scatter
    sample2 = datos[datos['host_years'].between(0, 12)].sample(
        min(4000, len(datos)), random_state=42)
    axes[1].scatter(sample2['host_years'], sample2['price_real'],
                    alpha=0.25, s=8, color='#9B59B6')
    axes[1].set_xlim(0, 12)
    axes[1].set_ylim(0, 500)
    axes[1].set_xlabel('Años del Anfitrión en la Plataforma')
    axes[1].set_ylabel('Precio Real ($)')
    axes[1].set_title('Dispersión: Antigüedad vs Precio')

    plt.tight_layout()
    plt.show()
else:
    print("Columna 'host_years' no disponible.")

In [ ]:
# ===========================================================================
# HEATMAP — REPUTACIÓN vs PRECIO (RESUMEN VISUAL)
# ===========================================================================

rep_cols = [c for c in ['host_has_profile_pic', 'host_identity_verified',
                         'instant_bookable', 'number_of_reviews',
                         'review_scores_rating', 'host_years', 'log_price']
            if c in datos.columns]

corr_rep = datos[rep_cols].corr()

plt.figure(figsize=(9, 6))
mask_rep = np.triu(np.ones_like(corr_rep, dtype=bool))
sns.heatmap(corr_rep, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask_rep, linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 10})
plt.title('Correlación: Variables de Reputación vs Precio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.1 Narrativa — Análisis de Reputación del Anfitrión

**¿Más reseñas → mayor precio?**
La correlación entre `number_of_reviews` y el precio es débil o incluso ligeramente negativa. Esto es contra-intuitivo pero tiene una explicación lógica: las propiedades más baratas tienden a tener **más rotación de huéspedes** (y por tanto más reseñas), mientras que las propiedades de lujo tienen menos reservas pero a precios más altos. Las reseñas no causan el precio; ambas son resultado de la popularidad y accesibilidad de la propiedad.

**¿Identidad verificada → mayor confianza y precio?**
Los anfitriones con identidad verificada y foto de perfil tienden a cobrar precios ligeramente mayores. Esto sugiere que la **señalización de confianza** permite a los anfitriones posicionarse mejor en el mercado y justificar un precio premium.

**¿Reserva instantánea → qué impacto tiene?**
Las propiedades con reserva instantánea muestran precios promedio ligeramente menores. Una posible explicación: los anfitriones que permiten reserva instantánea suelen estar más orientados al volumen que al precio.

**¿Calificación y precio?**
Existe una relación no lineal: las propiedades con calificaciones muy bajas (<80) tienen precios menores, pero entre calificaciones de 85-100 la diferencia de precio es pequeña. La calificación actúa más como un **umbral mínimo** (sin una buena calificación es difícil reservar) que como un diferenciador de precio en el rango alto.

**¿Antigüedad del anfitrión?**
Los anfitriones con más años en la plataforma tienden a cobrar precios levemente más altos, posiblemente porque han aprendido a optimizar sus precios y han construido su reputación con el tiempo.

# 10. MODELO EXTENDIDO: REGRESIÓN CON VARIABLES GEOGRÁFICAS Y DE REPUTACIÓN

Comparamos el modelo base (solo características físicas) con un modelo extendido que incluye variables de zona y reputación.

In [ ]:
# ===========================================================================
# PREPARAR VARIABLES PARA MODELO EXTENDIDO
# ===========================================================================

# Codificar room_type como dummies
if 'room_type' in datos.columns:
    room_dummies = pd.get_dummies(datos['room_type'], prefix='room', drop_first=True)
    datos = pd.concat([datos, room_dummies], axis=1)
    room_cols = room_dummies.columns.tolist()
else:
    room_cols = []

# Variables del modelo extendido
ext_features = (['accommodates', 'bedrooms', 'bathrooms', 'n_amenities'] +
                room_cols +
                [c for c in ['host_has_profile_pic', 'host_identity_verified',
                              'instant_bookable', 'review_scores_rating', 'host_years']
                 if c in datos.columns])

print("Variables del modelo extendido:", ext_features)

X_ext = datos[ext_features].copy().fillna(0)
y_ext = datos['log_price'].copy()

# Asegurar que todo sea numérico
X_ext = X_ext.apply(pd.to_numeric, errors='coerce').fillna(0)

# Eliminar NaN en y
mask_ext = y_ext.notna()
X_ext, y_ext = X_ext[mask_ext], y_ext[mask_ext]

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_ext, y_ext, test_size=0.2, random_state=42)

modelo_ext = LinearRegression()
modelo_ext.fit(X_tr_e, y_tr_e)
y_pred_ext = modelo_ext.predict(X_te_e)

r2_ext   = r2_score(y_te_e, y_pred_ext)
rmse_ext = np.sqrt(mean_squared_error(y_te_e, y_pred_ext))
r2_adj_ext = 1 - (1 - r2_ext) * (len(y_te_e)-1) / (len(y_te_e) - len(ext_features) - 1)

print("\n" + "=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)
print(f"{'Modelo':<35} {'R²':>8} {'R² Adj':>8} {'RMSE':>8}")
print("-" * 60)
print(f"{'Básico (propiedades físicas)':<35} {r2:>8.4f} {r2_adj:>8.4f} {rmse:>8.4f}")
print(f"{'Extendido (+ tipo hab + reputación)':<35} {r2_ext:>8.4f} {r2_adj_ext:>8.4f} {rmse_ext:>8.4f}")
print("=" * 60)

In [ ]:
# ===========================================================================
# IMPORTANCIA DE VARIABLES — MODELO EXTENDIDO
# ===========================================================================

imp_ext = pd.DataFrame({
    'Variable':    ext_features,
    'Coeficiente': modelo_ext.coef_,
    'Abs_Coef':    np.abs(modelo_ext.coef_)
}).sort_values('Abs_Coef', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_ext = ['#2ECC71' if c > 0 else '#E74C3C' for c in imp_ext['Coeficiente']]
ax.barh(imp_ext['Variable'], imp_ext['Coeficiente'], color=colors_ext, edgecolor='white', linewidth=1.2)
ax.axvline(x=0, color='black', linewidth=1.5, linestyle='--')
ax.set_title('Importancia de Variables — Modelo Extendido\n(Verde=sube el precio, Rojo=baja el precio)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Valor del Coeficiente (sobre log_price)')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# PREDICCIONES FINALES — MODELO EXTENDIDO
# ===========================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evaluación del Modelo Extendido', fontsize=13, fontweight='bold')

# Real vs Predicho
axes[0].scatter(y_te_e, y_pred_ext, alpha=0.2, s=8, color='#9B59B6')
mn = min(y_te_e.min(), y_pred_ext.min())
mx = max(y_te_e.max(), y_pred_ext.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Predicción perfecta')
axes[0].set_title(f'Real vs Predicho — Modelo Extendido (R²={r2_ext:.3f})')
axes[0].set_xlabel('log_price Real')
axes[0].set_ylabel('log_price Predicho')
axes[0].legend()

# Histograma de residuos
res_ext = y_te_e.values - y_pred_ext
sns.histplot(res_ext, kde=True, ax=axes[1], color='#9B59B6', bins=50)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Distribución de Residuos')
axes[1].set_xlabel('Residuo (Real − Predicho)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

# MSE final
mse_ext = mean_squared_error(y_te_e, y_pred_ext)
print(f"MSE  Modelo Extendido  : {mse_ext:.4f}")
print(f"RMSE Modelo Extendido  : {np.sqrt(mse_ext):.4f}")
print(f"Error en precio real ≈ : ${np.expm1(np.sqrt(mse_ext))*100:.0f}")

# 11. CONCLUSIONES FINALES

---

## Eficacia del Modelo de Regresión Lineal Múltiple

Se construyeron dos versiones del modelo:

| Modelo | Variables | R² | Interpretación |
|--------|-----------|-----|----------------|
| **Básico** | accommodates, bedrooms, bathrooms, beds, n_amenities | ~0.45 | Las características físicas explican ~45% del precio |
| **Extendido** | + tipo de habitación + reputación del anfitrión | ~0.58 | Añadir variables de contexto mejora el poder predictivo |

El R² moderado no es un fracaso del modelo; es un hallazgo: **el precio en Airbnb no se determina únicamente por las características físicas**, sino que está fuertemente influido por la ubicación específica (a nivel de calle), la calidad de las fotografías, la temporada y la reputación acumulada del anfitrión.

---

## Hallazgos del Análisis Geográfico

1. **Williamsburg (Brooklyn)** es el vecindario con más listados de NYC, dominando en todos los segmentos de precio. Su popularidad refleja la gentrificación de Brooklyn y la demanda de turistas que buscan la experiencia local.

2. **Manhattan concentra los precios más altos:** SoHo, Tribeca y el West Village tienen precios promedio 2-3 veces superiores al resto de los vecindarios.

3. **Existe una bifurcación de mercado:** o un vecindario es popular Y caro (Manhattan central), o es popular Y moderado (Brooklyn), o es nicho Y caro (zonas exclusivas residenciales). No hay barrios con baja popularidad y precios altos sostenidos.

---

## Hallazgos del Análisis de Reputación

1. **La foto de perfil y la verificación de identidad** se asocian con precios ligeramente más altos, pero el efecto es modesto (~5-10%). La confianza ayuda, pero no determina el precio.

2. **Las reseñas tienen una relación no lineal con el precio:** propiedades con menos de 5 reseñas pueden tener precios altos o bajos según el perfil del anfitrión. Más reseñas no necesariamente implican precio más alto; las propiedades económicas acumulan más reseñas por tener más huéspedes.

3. **La calificación actúa como umbral:** una calificación < 80 penaliza visiblemente el precio, pero entre 85 y 100 la diferencia es pequeña. La excelencia es el estándar mínimo, no un diferenciador.

4. **Los anfitriones con más años en la plataforma** cobran precios levemente más altos, probablemente porque han optimizado su listado, sus precios y han construido una base de clientes.

---

## Propuestas de Mejora

1. **Modelos no lineales:** Un Random Forest o Gradient Boosting capturaría interacciones entre variables (ej. barrio × tipo de habitación) que la regresión lineal no puede modelar.

2. **Variables de ubicación enriquecidas:** Codificar la zona geográfica como "distancia al centro de Manhattan" o añadir variables de barrio (tasa de criminalidad, densidad de atracciones turísticas) mejoraría el R² sustancialmente.

3. **Análisis de temporalidad:** Si el dataset incluye fechas, analizar la variación estacional del precio (verano vs invierno, eventos especiales en NYC) agregaría una dimensión muy relevante al análisis.

4. **Análisis de texto de amenities:** La columna `amenities` contiene listas detalladas de servicios. Un análisis de frecuencia de amenidades específicas (piscina, terraza, vista al río) podría revelar qué servicios tienen el mayor impacto en el precio.

In [ ]:
# ===========================================================================
# EXPORTACIÓN FINAL DEL DATASET PROCESADO
# ===========================================================================
datos.to_csv("airbnb_procesado.csv", index=False)
print("Dataset listo para modelado adicional.")
print(f"Forma final: {datos.shape[0]:,} registros × {datos.shape[1]} columnas")
print("\nColumnas disponibles:")
print([c for c in datos.columns])

# RANDOM FOREST 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# ===========================================================================
# CONSTRUCCIÓN Y ENTRENAMIENTO DEL MODELO ROBUSTO
# ===========================================================================

# Cambiamos LinearRegression por RandomForestRegressor
# Este modelo es mucho más estable para una App Web
modelo = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
modelo.fit(X_train, y_train)

print("=" * 50)
print("MODELO RANDOM FOREST ENTRENADO")
print("=" * 50)

# Importancia de las variables (Sustituye a los coeficientes)
importancia_df = pd.DataFrame({
    'Variable': feature_cols,
    'Importancia': modelo.feature_importances_
}).sort_values('Importancia', ascending=False)

print(importancia_df.to_string(index=False))

# EXPORTACIÓN DEL MODELO

In [ ]:
# ==============================
# EXPORTAR MODELO
# ==============================
import joblib
import os

# Asegurarnos de que la carpeta existe
if not os.path.exists('../models'):
    os.makedirs('../models')

# CORRECTO: Guardamos los objetos entrenados, no los resultados
objetos_modelo = {
    'modelo': modelo,               # El objeto LinearRegression entrenado
    'features': feature_cols       # Si no usaste un StandardScaler/MinMaxScaler, pon None
}

# Guardar el archivo
joblib.dump(objetos_modelo, '../models/modelo_airbnb.joblib', compress=3)

print("¡Cerebro guardado correctamente! El archivo ya contiene el modelo listo para predecir.")